# Calibración de las distancias máximas y mínimas con la función de profundidad de la RealSense

Este módulo es una versión avanzada del proceso de calibración, implementando telemetría híbrida tridimensional. Toma el wireframe capturado sobre la mano humana mediante MediaPipe (2D) y le inyecta la profundidad real en milímetros obtenida del lente infrarrojo de la cámara Intel RealSense (3D). 

Al desproyectar estos píxeles utilizando los parámetros intrínsecos de la cámara (`rs2_deproject_pixel_to_point`), las distancias **mínimas** y **máximas** calculadas para cada Grado de Libertad (DOF) se obtienen estrictamente en **metros reales**, eliminando las distorsiones de escala, la fluctuación del eje Z y la dependencia de la perspectiva.

Estos valores métricos son estrictamente necesarios para el proceso matemático de **normalización** que transforma el movimiento físico humano en un vector proporcional del $0$ al $1000$ (Modbus TCP) para la mano robótica RH56DFTP-2R.

**Flujo**
1. **Inicialización**: Cargar las dependencias, encender ambos flujos de la cámara Intel RealSense (color y profundidad) y alinear espacialmente sus matrices para que encajen pixel a pixel.
2. **Extracción Intrínseca**: Extraer los parámetros ópticos de fábrica del lente infrarrojo para habilitar la desproyección matemática.
3. **Captura Interactiva**: Ejecutar la celda de calibración y seguir las instrucciones en pantalla, presionando la tecla `c` en 3 fases secuenciales:
   - *Fase 1*: Mano completamente abierta (Captura el vector $D_{max}$ para los 6 motores).
   - *Fase 2*: Puño cerrado normal (Captura el vector $D_{min}$ exclusivamente para los dedos principales 0 al 4).
   - *Fase 3*: Pulgar rotado hacia la palma (Captura el vector $D_{min}$ exclusivamente para el motor 5, evitando colisiones mecánicas).
4. **Exportación de Datos**: Copiar los vectores de calibración resultantes (`D_max` y `D_min`) impresos en consola, necesarios para el código de teleoperación futuro.

**Notas**
- MediaPipe asume imagen tipo espejo (selfie). Como la RealSense no refleja la imagen, la etiqueta `Right` puede corresponder a la mano **izquierda** real. Como sucede esto, se cambia el filtro a `"Left"`.
- **Desproyección Geométrica:** El bloque de extracción toma las coordenadas $(X, Y)$ de MediaPipe, el valor $(Z)$ del láser infrarrojo y, utilizando la función matemática nativa de `pyrealsense2`, genera un vector $(X, Y, Z)$ donde todas las variables operan en metros físicos, resolviendo el problema de mezclar píxeles con profundidades.
- **Wireframe Personalizado:** Para asegurar la correspondencia visual con la estructura mecánica, el bloque de dibujo omite la malla de 21 puntos y traza exclusivamente los 11 nodos de interés (bases y yemas), separando por color la estructura base (muñeca y palma) de las distancias calculadas.
- **Acotamiento del Lente (Clamping):** Se incluye un límite de seguridad matemático sobre los píxeles extraídos (`max`/`min`) para garantizar que MediaPipe nunca intente leer un píxel fuera del hardware físico (ej. el píxel 640 en una cámara de 640x480).
---

# Instrucciones del proceso de calibración

1. Coloca tu **mano derecha** frente a la cámara **completamente abierta**.
2. Presiona la tecla `c` para capturar las distancias **máximas**.
3. Luego, cierra la mano en **puño totalmente cerrado** (con el pulgar únicamente **flexionado** al máximo contra el **dedo índice**).
4. Presiona la tecla `c`nuevamente para capturar las distancias **mínimas**.
5. Ahora, abre nuevamente la mano y lleva **solamente** el pulgar hacia la palma, además ubícala de **manera lateral**, de modo que el meñique sea el dedo más cercano a la cámara, seguido por yema del dedo pulgar.
6. Presiona la tecla `c`otra vesz para capturar la distancia **mínima** correspondiente a la **rotación del pulgar**
7. Presiona `q`para salir en cualquier momento.

# 1. Imports y configuración de MediaPipe + RealSense

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp
import warnings

# Silenciar advertencia de Protobuf
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

# 1. Inicializar MediaPipe y configuración base
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

# 2. Inicializar Intel RealSense (Color + Profundidad)
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

# Alinear Profundidad a Color
align = rs.align(rs.stream.color)
pipeline.start(config)

# Extraer parámetros intrínsecos de la cámara
perfil_camara = pipeline.get_active_profile()
perfil_profundidad = perfil_camara.get_stream(rs.stream.depth).as_video_stream_profile()
intrinsics = perfil_profundidad.get_intrinsics()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792528627.494522   21089 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792528627.514889   21088 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


# 2. Ejecución de la calibración

In [2]:
# Variables de estado y buffers para las 3 medidas manuales por fase
modo = "MÁXIMA"
muestras_maxima = []
muestras_min_dedos = []
muestras_min_pulgar = []

D_max = np.zeros(6)
D_min = np.zeros(6)

print("=== CALIBRACIÓN HÍBRIDA 3D EN 3 FASES (3 MEDIDAS MANUALES POR FASE) ===")
print("Fase 1: Coloca tu mano COMPLETAMENTE ABIERTA y presiona 'c' 3 veces (tomando 3 medidas).")

try:
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame: 
            continue
        
        color_image = np.asanyarray(color_frame.get_data())
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
        
        results = hands.process(image_rgb)
        
        D_actual = np.zeros(6)
        mano_detectada = False
        profundidad_valida = True

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                if handedness.classification[0].label == "Left":
                    mano_detectada = True
                    
                    h, w, _ = color_image.shape
                    nodos_interes = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                    puntos_3d = {}
                    pixeles_dibujo = {}
                    
                    for idx in nodos_interes:
                        lm = hand_landmarks.landmark[idx]
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        cx = max(0, min(cx, w - 1))
                        cy = max(0, min(cy, h - 1))
                        
                        pixeles_dibujo[idx] = (cx, cy)
                        z_metros = depth_frame.get_distance(cx, cy)
                        
                        if z_metros > 0:
                            pto_metrico = rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros)
                            puntos_3d[idx] = np.array(pto_metrico)
                        else:
                            profundidad_valida = False
                            #break

                    if profundidad_valida:
                        d_indice  = np.linalg.norm(puntos_3d[8] - puntos_3d[5])
                        d_corazon = np.linalg.norm(puntos_3d[12] - puntos_3d[9])
                        d_anular  = np.linalg.norm(puntos_3d[16] - puntos_3d[13])
                        d_menique = np.linalg.norm(puntos_3d[20] - puntos_3d[17])
                        d_pulgar_flex = np.linalg.norm(puntos_3d[4] - puntos_3d[5])
                        d_pulgar_rot  = np.linalg.norm(puntos_3d[4] - puntos_3d[17])
                        
                        D_actual = np.array([d_menique, d_anular, d_corazon, d_indice, d_pulgar_flex, d_pulgar_rot])

                    # --- DIBUJO DEL WIREFRAME PERSONALIZADO (11 Nodos) ---
                    for idx, (cx, cy) in pixeles_dibujo.items():
                        cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)

                    cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[5], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[9], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[13], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[17], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[2], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles_dibujo[2], pixeles_dibujo[4], (200, 200, 200), 2)

                    cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[8], (0, 255, 0), 3)   # Índice
                    cv2.line(color_image, pixeles_dibujo[9], pixeles_dibujo[12], (0, 255, 0), 3)  # Corazón
                    cv2.line(color_image, pixeles_dibujo[13], pixeles_dibujo[16], (0, 255, 0), 3) # Anular
                    cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[20], (0, 255, 0), 3) # Meñique
                    cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[4], (0, 255, 255), 3) # Pulgar Flexión
                    cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[4], (255, 0, 255), 3)# Pulgar Rotación

                    # Textos en pantalla según la fase y número de medida actual
                    if modo == "MÁXIMA":
                        txt = f"1. Abierta - Medida {len(muestras_maxima)+1}/3 (Presiona 'c')"
                        cv2.putText(color_image, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    elif modo == "MÍNIMA_DEDOS":
                        txt = f"2. Puno Cerrado - Medida {len(muestras_min_dedos)+1}/3 (Presiona 'c')"
                        cv2.putText(color_image, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    elif modo == "MÍNIMA_PULGAR":
                        txt = f"3. Solo Pulgar - Medida {len(muestras_min_pulgar)+1}/3 (Presiona 'c')"
                        cv2.putText(color_image, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
                    elif modo == "LISTO":
                        cv2.putText(color_image, "Calibracion Completa. Presiona 'q' para salir.", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        cv2.imshow('Calibracion Hibrida 3D - 3 Medidas', color_image)

        # --- LÓGICA DE CAPTURA MANUAL (3 MEDIDAS POR FASE) ---
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and mano_detectada and profundidad_valida:
            if modo == "MÁXIMA":
                muestras_maxima.append(D_actual.copy())
                print(f"✓ Medida Máxima {len(muestras_maxima)}/3 capturada.")
                if len(muestras_maxima) == 3:
                    D_max = np.mean(np.array(muestras_maxima), axis=0)
                    print(f"\n✓ Promedio MÁXIMO calculado:\n{D_max}")
                    print("\n-> AHORA CIERRA EL PUÑO NORMAL, TOMA 3 MEDIDAS Y PRESIONA 'c' EN CADA UNA.")
                    modo = "MÍNIMA_DEDOS"
                
            elif modo == "MÍNIMA_DEDOS":
                muestras_min_dedos.append(D_actual.copy())
                print(f"✓ Medida Mínima Dedos {len(muestras_min_dedos)}/3 capturada.")
                if len(muestras_min_dedos) == 3:
                    promedio_dedos = np.mean(np.array(muestras_min_dedos), axis=0)
                    D_min[0:5] = promedio_dedos[0:5]
                    print(f"\n✓ Promedio MÍNIMO (Dedos) calculado:\n{D_min[0:5]}")
                    print("\n-> AHORA ESTIRA LOS DEDOS Y LLEVA EL PULGAR SOLO HACIA LA PALMA (Rotación), TOMA 3 MEDIDAS Y PRESIONA 'c'.")
                    modo = "MÍNIMA_PULGAR"
                
            elif modo == "MÍNIMA_PULGAR":
                muestras_min_pulgar.append(D_actual.copy())
                print(f"✓ Medida Mínima Pulgar {len(muestras_min_pulgar)}/3 capturada.")
                if len(muestras_min_pulgar) == 3:
                    promedio_pulgar = np.mean(np.array(muestras_min_pulgar), axis=0)
                    D_min[5] = promedio_pulgar[5]
                    
                    print("\n=== CALIBRACIÓN EXITOSA (PROMEDIADA DE 3 MEDIDAS MANUALES) ===")
                    print("Copia y pega estos arreglos en tu script:")
                    print(f"D_max = np.array([{D_max[0]:.4f}, {D_max[1]:.4f}, {D_max[2]:.4f}, {D_max[3]:.4f}, {D_max[4]:.4f}, {D_max[5]:.4f}])")
                    print(f"D_min = np.array([{D_min[0]:.4f}, {D_min[1]:.4f}, {D_min[2]:.4f}, {D_min[3]:.4f}, {D_min[4]:.4f}, {D_min[5]:.4f}])")
                    modo = "LISTO"
                
        elif key == ord('q'):
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    print("Cámara apagada y ventanas cerradas.")

=== CALIBRACIÓN HÍBRIDA 3D EN 3 FASES (3 MEDIDAS MANUALES POR FASE) ===
Fase 1: Coloca tu mano COMPLETAMENTE ABIERTA y presiona 'c' 3 veces (tomando 3 medidas).
✓ Medida Máxima 1/3 capturada.
✓ Medida Máxima 2/3 capturada.
✓ Medida Máxima 3/3 capturada.

✓ Promedio MÁXIMO calculado:
[0.10018661 0.12249904 0.12952277 0.12131454 0.11337251 0.19521745]

-> AHORA CIERRA EL PUÑO NORMAL, TOMA 3 MEDIDAS Y PRESIONA 'c' EN CADA UNA.
✓ Medida Mínima Dedos 1/3 capturada.
✓ Medida Mínima Dedos 2/3 capturada.
✓ Medida Mínima Dedos 3/3 capturada.

✓ Promedio MÍNIMO (Dedos) calculado:
[0.02809702 0.03306664 0.02743163 0.02674754 0.06120138]

-> AHORA ESTIRA LOS DEDOS Y LLEVA EL PULGAR SOLO HACIA LA PALMA (Rotación), TOMA 3 MEDIDAS Y PRESIONA 'c'.
✓ Medida Mínima Pulgar 1/3 capturada.
✓ Medida Mínima Pulgar 2/3 capturada.
✓ Medida Mínima Pulgar 3/3 capturada.

=== CALIBRACIÓN EXITOSA (PROMEDIADA DE 3 MEDIDAS MANUALES) ===
Copia y pega estos arreglos en tu script:
D_max = np.array([0.1002, 0.1225, 0.12